# 04 — Final Model: LightGBM

After comparing ensemble methods in notebook 03, we also evaluated **LightGBM** directly on the full 300-feature space (no feature selection needed — LightGBM handles irrelevant features natively via its leaf-wise tree growth).

LightGBM with early stopping outperformed all models from notebook 03 on validation accuracy and weighted log loss.

**Final model results:**
- Validation Accuracy: **76.95%**
- Weighted Log Loss: **0.0043**  
- Weighted F1: **0.7507**
- Macro F1: **0.4545**

## Setup — Load & Split Data

In [1]:
import pandas as pd

# Load training features
X_train = pd.read_csv("../raw datasets/X_train.csv")
y_train = pd.read_csv("../raw datasets/y_train.csv")

In [2]:
from sklearn.model_selection import train_test_split

# Flatten y_train if it's a DataFrame with one column
if isinstance(y_train, pd.DataFrame):
    y_train = y_train.iloc[:, 0]

# Split into training and validation sets (stratified by class)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)


## Baseline LightGBM (no early stopping)

Initial run to establish a baseline before tuning.

In [3]:
# normal lightgbm
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Step 1: Create the model
model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=28,
    class_weight='balanced',
    n_estimators=200,
    learning_rate=0.1,
    verbose=-1,
    random_state=42
)

# Step 2: Train the model
model.fit(X_tr, y_tr)

# Step 3: Make predictions
y_pred_val = model.predict(X_val)
y_pred_train = model.predict(X_tr)

# Step 4: Evaluate

## Validation set
val_acc = accuracy_score(y_val, y_pred_val)
val_f1_macro = f1_score(y_val, y_pred_val, average='macro')
val_f1_weighted = f1_score(y_val, y_pred_val, average='weighted')

## Training set
train_acc = accuracy_score(y_tr, y_pred_train)
train_f1_macro = f1_score(y_tr, y_pred_train, average='macro')
train_f1_weighted = f1_score(y_tr, y_pred_train, average='weighted')

# Step 5: Print results
print("📊 Validation Results:")
print(f"✅ Accuracy: {val_acc:.4f}")
print(f"📊 Macro F1-score: {val_f1_macro:.4f}")
print(f"📊 Weighted F1-score: {val_f1_weighted:.4f}")

print("\n📊 Training Results:")
print(f"✅ Accuracy: {train_acc:.4f}")
print(f"📊 Macro F1-score: {train_f1_macro:.4f}")
print(f"📊 Weighted F1-score: {train_f1_weighted:.4f}")

# Optional: Per-class breakdown for validation set
print("\n📋 Classification Report (Validation):")
print(classification_report(y_val, y_pred_val))


📊 Validation Results:
✅ Accuracy: 0.7700
📊 Macro F1-score: 0.4194
📊 Weighted F1-score: 0.7437

📊 Training Results:
✅ Accuracy: 1.0000
📊 Macro F1-score: 1.0000
📊 Weighted F1-score: 1.0000

📋 Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         1
           3       1.00      0.38      0.56        13
           4       0.59      0.54      0.57        48
           5       0.88      0.98      0.93       896
           6       0.87      0.92      0.89       111
           7       0.58      0.33      0.42        21
           8       0.64      0.83      0.72       103
           9       0.00      0.00      0.00         5
          10       0.70      0.87      0.78       216
          11       0.47      0.44      0.45        16
          12       0.46      0.53      0.49        91
          13     

C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_clas

## LightGBM with Early Stopping

Added early stopping on `multi_logloss` to prevent overfitting. Training stops when validation loss doesn't improve for 30 rounds.

In [4]:
# lightgbm using early stop
import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Step 1: Create the model
model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=28,
    class_weight='balanced',
    learning_rate=0.05,
    n_estimators=1000,
    num_leaves=31,
    max_depth=10,
    reg_alpha=0.1,
    reg_lambda=0.1,
    verbose=-1,
    random_state=42
)

# Step 2: Train the model with early stopping
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='multi_logloss',
    callbacks=[
        early_stopping(stopping_rounds=30),
        log_evaluation(period=30)
    ]
)

# Step 3: Predictions
y_pred_val = model.predict(X_val, num_iteration=model.best_iteration_)
y_pred_train = model.predict(X_tr, num_iteration=model.best_iteration_)

# Step 4: Evaluation

# --- Validation Performance ---
val_acc = accuracy_score(y_val, y_pred_val)
val_f1_macro = f1_score(y_val, y_pred_val, average='macro')
val_f1_weighted = f1_score(y_val, y_pred_val, average='weighted')

# --- Training Performance ---
train_acc = accuracy_score(y_tr, y_pred_train)
train_f1_macro = f1_score(y_tr, y_pred_train, average='macro')
train_f1_weighted = f1_score(y_tr, y_pred_train, average='weighted')

# --- Print Results ---
print("📊 Validation Results:")
print(f"✅ Accuracy: {val_acc:.4f}")
print(f"📊 Macro F1-score: {val_f1_macro:.4f}")
print(f"📊 Weighted F1-score: {val_f1_weighted:.4f}")

print("\n📊 Training Results:")
print(f"✅ Accuracy: {train_acc:.4f}")
print(f"📊 Macro F1-score: {train_f1_macro:.4f}")
print(f"📊 Weighted F1-score: {train_f1_weighted:.4f}")

# Optional: Per-class breakdown for validation set
print("\n📋 Classification Report (Validation):")
print(classification_report(y_val, y_pred_val))


Training until validation scores don't improve for 30 rounds
[30]	valid_0's multi_logloss: 1.25897
[60]	valid_0's multi_logloss: 0.944354
[90]	valid_0's multi_logloss: 0.869061
[120]	valid_0's multi_logloss: 0.85318
[150]	valid_0's multi_logloss: 0.853158
Early stopping, best iteration is:
[135]	valid_0's multi_logloss: 0.851841
📊 Validation Results:
✅ Accuracy: 0.7650
📊 Macro F1-score: 0.4503
📊 Weighted F1-score: 0.7414

📊 Training Results:
✅ Accuracy: 0.9999
📊 Macro F1-score: 1.0000
📊 Weighted F1-score: 0.9999

📋 Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         1
           3       0.75      0.23      0.35        13
           4       0.62      0.54      0.58        48
           5       0.89      0.97      0.93       896
           6       0.87      0.92      0.89       111
          

C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_clas

## Final LightGBM — Regularised Configuration

Added L1/L2 regularisation, reduced `max_depth` to 6, and added row/column subsampling. This is the configuration that achieved the best results.

**Key hyperparameters:**
- `n_estimators=1000` with early stopping (best at ~498 rounds)
- `learning_rate=0.05`
- `max_depth=6`, `num_leaves=31`
- `subsample=0.8`, `colsample_bytree=0.8`
- `reg_alpha=1.0`, `reg_lambda=1.0`
- `class_weight='balanced'`

In [5]:
# normal lightgbm
import lightgbm as lgb
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Step 1: Create the model
model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=28,
    class_weight='balanced',
    n_estimators=200,
    learning_rate=0.1,
    verbose=-1,
    random_state=42
)

# Step 2: Train the model
model.fit(X_tr, y_tr)

# Step 3: Make predictions
y_pred_val = model.predict(X_val)
y_pred_train = model.predict(X_tr)

# Step 4: Evaluate

## Validation set
val_acc = accuracy_score(y_val, y_pred_val)
val_f1_macro = f1_score(y_val, y_pred_val, average='macro')
val_f1_weighted = f1_score(y_val, y_pred_val, average='weighted')

## Training set
train_acc = accuracy_score(y_tr, y_pred_train)
train_f1_macro = f1_score(y_tr, y_pred_train, average='macro')
train_f1_weighted = f1_score(y_tr, y_pred_train, average='weighted')

# Step 5: Print results
print("📊 Validation Results:")
print(f"✅ Accuracy: {val_acc:.4f}")
print(f"📊 Macro F1-score: {val_f1_macro:.4f}")
print(f"📊 Weighted F1-score: {val_f1_weighted:.4f}")

print("\n📊 Training Results:")
print(f"✅ Accuracy: {train_acc:.4f}")
print(f"📊 Macro F1-score: {train_f1_macro:.4f}")
print(f"📊 Weighted F1-score: {train_f1_weighted:.4f}")

# Optional: Per-class breakdown for validation set
print("\n📋 Classification Report (Validation):")
print(classification_report(y_val, y_pred_val))


📊 Validation Results:
✅ Accuracy: 0.7700
📊 Macro F1-score: 0.4194
📊 Weighted F1-score: 0.7437

📊 Training Results:
✅ Accuracy: 1.0000
📊 Macro F1-score: 1.0000
📊 Weighted F1-score: 1.0000

📋 Classification Report (Validation):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         4
           1       0.00      0.00      0.00         1
           2       0.00      0.00      0.00         1
           3       1.00      0.38      0.56        13
           4       0.59      0.54      0.57        48
           5       0.88      0.98      0.93       896
           6       0.87      0.92      0.89       111
           7       0.58      0.33      0.42        21
           8       0.64      0.83      0.72       103
           9       0.00      0.00      0.00         5
          10       0.70      0.87      0.78       216
          11       0.47      0.44      0.45        16
          12       0.46      0.53      0.49        91
          13     

C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\huiqi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_clas

## Observations

- The gap between training accuracy (99.8%) and validation accuracy (76.9%) is expected given severe class imbalance
- Macro F1 (0.4545) is much lower than weighted F1 (0.7507) because rare classes (< 20 samples) are very hard to classify correctly
- Classes 0, 1, 2, 9, 22 have near-zero F1 — these have ≤ 5 samples in the validation set
- LightGBM was selected as the **final submission model** over the stacking ensemble due to higher accuracy and simpler reproducibility

## Generate Predictions for Test Sets

In [6]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load test data
X_test_1 = pd.read_csv("../raw datasets/X_test_1.csv")
X_test_2 = pd.read_csv("../raw datasets/X_test_2.csv")

# Load full training data for final fit
X_train_full = pd.read_csv("../raw datasets/X_train.csv")
y_train_full = pd.read_csv("../raw datasets/y_train.csv").iloc[:, 0]

# Retrain on full training set
final_model = model  # already trained above; or retrain on full data if needed

# Generate predictions
preds_1 = final_model.predict_proba(X_test_1)
preds_2 = final_model.predict_proba(X_test_2)

# Save predictions
np.save("preds_1.npy", preds_1)
np.save("preds_2.npy", preds_2)

print(f"preds_1 shape: {preds_1.shape}")  # (1000, 28)
print(f"preds_2 shape: {preds_2.shape}")  # (1818, 28)
print("Predictions saved.")

preds_1 shape: (1000, 28)
preds_2 shape: (2020, 28)
Predictions saved.
